# 01 — Exploration et qualification des sources

**Benoit Girard** — projet n°12, *Extrayez des données multimodales de sites web*

Ce notebook accompagne le livrable n°1. Il vérifie **concrètement**, source par source, que les données annoncées sont bien là : un texte, une image, et le cas échéant un label. Le raisonnement complet — pourquoi ces sources, pourquoi pas de scraping — est dans `docs/rapport_exploration_sources.md`.

In [1]:
import sys
from pathlib import Path

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE / "src"))

from dotenv import load_dotenv

load_dotenv(RACINE / ".env")

from checkitai.logging_setup import setup_logging

setup_logging()

## Les quatre sources et leurs méthodes d'accès

Une source du pipeline, c'est un module dans `checkitai.sources` exposant une fonction `fetch_*`, et une ligne dans la table des connecteurs. Chacune passe par un canal différent, et c'est voulu : cela évite de dépendre d'un seul mode d'accès.

In [2]:
from checkitai.extract import _CONNECTEURS

for nom, fonction in _CONNECTEURS.items():
    print(f"{nom:18s} -> {fonction.__module__}")

rss                -> checkitai.sources.rss
newsdata           -> checkitai.sources.newsdata
fakenewsnet        -> checkitai.sources.fakenewsnet
kaggle_fakeddit    -> checkitai.sources.kaggle_fakeddit


## 1. Flux RSS — le socle multimodal

Les flux RSS sont publiés par les éditeurs *pour être rediffusés* : gratuits, sans clé, mis à jour en continu. La difficulté est ailleurs — **l'image n'est jamais au même endroit** selon l'éditeur. Le connecteur la cherche successivement dans `media:content`, `media:thumbnail`, les pièces jointes, puis dans le HTML du résumé.

In [3]:
from checkitai.config import ExtractionConfig
from checkitai.sources.rss import fetch_rss_feed

config = ExtractionConfig()
flux = dict(config.rss_feeds)
publications = fetch_rss_feed("bbc_news", flux["bbc_news"], config)

avec_image = [p for p in publications if p["image_url"]]
print(f"{len(publications)} publications lues, dont {len(avec_image)} avec une image")

exemple = avec_image[0]
for cle in ("title", "url", "image_url", "access_method"):
    print(f"{cle:14s}: {str(exemple[cle])[:88]}")

2026-08-20 11:52:59 | INFO    | checkitai.sources.rss | RSS : lecture du flux 'bbc_news' (https://feeds.bbci.co.uk/news/world/rss.xml)


2026-08-20 11:53:00 | INFO    | checkitai.sources.rss | RSS : 27 publications recuperees depuis 'bbc_news'


27 publications lues, dont 27 avec une image
title         : At least 13 killed in Kyiv as Ukraine grapples with air defence shortages
url           : https://www.bbc.co.uk/news/articles/c98vzmden5yo?at_medium=RSS&at_campaign=rss
image_url     : https://ichef.bbci.co.uk/ace/standard/240/cpsprodpb/630f/live/b3d19af0-9c6e-11f1-8efc-bf
access_method : flux_rss


## 2. API NewsData.io — actualité déjà normalisée

L'API renvoie du JSON avec un champ `image_url` explicite. Elle impose en échange un quota journalier : le connecteur ne lit qu'une page, et **se désactive proprement** si aucune clé n'est fournie plutôt que de faire échouer le pipeline.

In [4]:
from checkitai.sources import newsdata

print("Clé API disponible :", newsdata.is_enabled())
articles = newsdata.fetch_newsdata(config)
print(f"{len(articles)} articles récupérés")
if articles:
    print("Titre :", articles[0]["title"][:88])
    print("Image :", articles[0]["image_url"][:88])

Clé API disponible : True
2026-08-20 11:53:00 | INFO    | checkitai.sources.newsdata | NewsData.io : appel de l'API (https://newsdata.io/api/1/news)


2026-08-20 11:53:00 | INFO    | checkitai.sources.newsdata | NewsData.io : 10 articles recuperes


10 articles récupérés
Titre : Dead beaver in Montgomery County tests positive for rabies
Image : https://bloximages.newyork1.vip.townnews.com/fredericknewspost.com/content/tncms/assets/


## 3. FakeNewsNet — un jeu labellisé, mais sans image

Les CSV publiés sur GitHub contiennent `id, news_url, title, tweet_ids`. Il n'y a **aucune image** : ce que la fiche du jeu de données ne dit pas explicitement.

Deux conséquences pratiques traitées par le connecteur :
1. la colonne `tweet_ids` dépasse la taille de champ acceptée par défaut par le module `csv` — il faut relever la limite, sinon la lecture échoue ;
2. l'image doit être retrouvée ailleurs : dans la balise `og:image` que l'éditeur publie lui-même sur la page de l'article.

In [5]:
from checkitai.sources import fakenewsnet

chemin = fakenewsnet.telecharge_csv("politifact_fake.csv", config)
lignes = fakenewsnet.lit_csv(chemin)
print("Colonnes réelles du fichier :", list(lignes[0].keys()))
print(f"{len(lignes)} lignes labellisées disponibles")
print("Exemple de titre :", lignes[0]["title"][:88])

2026-08-20 11:53:00 | INFO    | checkitai.sources.fakenewsnet | FakeNewsNet : 'politifact_fake.csv' déjà en cache


Colonnes réelles du fichier : ['id', 'news_url', 'title', 'tweet_ids']
432 lignes labellisées disponibles
Exemple de titre : BREAKING: First NFL Team Declares Bankruptcy Over Kneeling Thugs


### Le rendement de l'enrichissement Open Graph

On mesure ce que l'on récupère réellement : les URL de PolitiFact datent de 2016-2018 et beaucoup ne répondent plus. C'est une contrainte à connaître, pas un défaut à cacher — les publications sans image seront écartées à la transformation.

In [6]:
from checkitai.sources import opengraph

echantillon = [fakenewsnet._construit_record(ligne, "politifact", "fake") for ligne in lignes[:10]]
compteurs = opengraph.enrichit_publications(echantillon, config)
print(compteurs)

2026-08-20 11:53:23 | INFO    | checkitai.sources.opengraph | Open Graph : 3 images retrouvées sur 10 articles consultés


{'tentees': 10, 'trouvees': 3}


## 4. Fakeddit — le jeu multimodal de Kaggle

Fakeddit associe nativement un titre et une image, avec trois niveaux de labels. Le fichier se télécharge une fois à la main depuis Kaggle et se dépose dans `data/raw/kaggle/` ; en son absence, le connecteur lit un échantillon de démonstration versionné, de structure identique.

In [7]:
import pandas as pd
from checkitai.sources import kaggle_fakeddit

publications_kaggle = kaggle_fakeddit.fetch_fakeddit(config)
df_kaggle = pd.DataFrame(publications_kaggle)
print(f"{len(df_kaggle)} publications chargées")
df_kaggle["label"].value_counts()

2026-08-20 11:53:25 | INFO    | checkitai.sources.kaggle_fakeddit | Fakeddit : jeu Kaggle absent, utilisation de fakeddit_sample.tsv


2026-08-20 11:53:25 | INFO    | checkitai.sources.kaggle_fakeddit | Fakeddit : 24 publications chargées depuis fakeddit_sample.tsv (échantillon de démonstration)


24 publications chargées


label
fake    12
real    12
Name: count, dtype: int64

## Bilan

Les quatre sources sont complémentaires : les flux RSS et l'API apportent le **volume et la fraîcheur**, FakeNewsNet et Fakeddit apportent des **labels**. Aucune ne passe par du scraping : ce sont toutes des canaux que le producteur de la donnée a prévus pour cet usage.